# GANs

Train a small generator and discriminator on a one-dimensional Gaussian target. The purpose is to inspect alternating updates, not produce a high-quality generative model.

In [ ]:
import torch
from torch import nn

torch.manual_seed(39)
torch.set_num_threads(1)
print('CPU | tiny one-dimensional GAN')

In [ ]:
generator = nn.Sequential(nn.Linear(1, 16), nn.Tanh(), nn.Linear(16, 1))
discriminator = nn.Sequential(nn.Linear(1, 16), nn.LeakyReLU(0.2), nn.Linear(16, 1))
g_optimizer = torch.optim.Adam(generator.parameters(), lr=0.01)
d_optimizer = torch.optim.Adam(discriminator.parameters(), lr=0.01)
loss_fn = nn.BCEWithLogitsLoss()

In [ ]:
for _ in range(100):
    real = torch.randn(64, 1) + 2
    noise = torch.randn(64, 1)
    fake = generator(noise).detach()
    d_loss = loss_fn(discriminator(real), torch.ones(64, 1)) + loss_fn(discriminator(fake), torch.zeros(64, 1))
    d_optimizer.zero_grad()
    d_loss.backward()
    d_optimizer.step()

    noise = torch.randn(64, 1)
    generated = generator(noise)
    g_loss = loss_fn(discriminator(generated), torch.ones(64, 1))
    g_optimizer.zero_grad()
    g_loss.backward()
    g_optimizer.step()
with torch.no_grad():
    samples = generator(torch.randn(256, 1)).squeeze(1)
assert torch.isfinite(g_loss) and samples.shape == (256,)
print(f'discriminator loss={d_loss.item():.4f} | generator loss={g_loss.item():.4f}')
print(f'generated mean={samples.mean().item():.3f}, target mean=2.000')

This short run is intentionally not a success claim: its generated mean is about 0.872, far from the target mean of 2.0. GAN training can be unstable, and a finite loss does not show that the generator learned the target distribution. Increase training or change the setup only after inspecting generated samples and validation measures.